Next Generation Sequencing

In [3]:
# Generate a dummy FASTQ file with one sequence
dummy_fastq = """@SEQ_ID_001_Illumina_Run
GATTTGGGGTTCAAAGCAGTATCGATCAAATAGTAAATCCATTTGTTCAACTCACAGTTT
+
!''*((((***+))%%%++)(%%%%).1***-+*''))**55CCF>>>>>>CCCCCCC65"""

with open("sample_read.fastq", "w") as f:
    f.write(dummy_fastq)
    
print("Dummy FASTQ created!")

Dummy FASTQ created!


Phred Score

In [4]:
from Bio import SeqIO

# Parse the FASTQ file just like we did with FASTA
for record in SeqIO.parse("sample_read.fastq", "fastq"):
    print(f"ID: {record.id}")
    print(f"Sequence: {record.seq[:20]}...") # Print first 20 bases
    
    # Extract the quality scores as actual numbers!
    quality_scores = record.letter_annotations["phred_quality"]
    
    print(f"Quality Scores: {quality_scores[:20]}...")

ID: SEQ_ID_001_Illumina_Run
Sequence: GATTTGGGGTTCAAAGCAGT...
Quality Scores: [0, 6, 6, 9, 7, 7, 7, 7, 9, 9, 9, 10, 8, 8, 4, 4, 4, 10, 10, 8]...


FASTQ Quality Control & Read Trimming

Phred Score

In [9]:
# Generate a dummy FASTQ file with a degrading read
# 'I' represents a perfect score (Q40), '!' is terrible (Q0)
dummy_bad_fastq = """@Read_001_Degraded_Tail
ATGCGTACGTTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCT
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII555555!!!!!"""

with open("degraded_read.fastq", "w") as f:
    f.write(dummy_bad_fastq)
    
print("Degraded FASTQ created!")

Degraded FASTQ created!


In [10]:
from Bio import SeqIO

# 1. Define our quality threshold
THRESHOLD = 20

# 2. Open and parse the FASTQ file
for record in SeqIO.parse("degraded_read.fastq", "fastq"):
    
    # Extract the numerical Phred scores
    quals = record.letter_annotations["phred_quality"]
    
    # 3. Find the exact index where the quality drops
    cut_index = len(record) # Default to keeping the whole read
    
    for i, score in enumerate(quals):
        if score < THRESHOLD:
            cut_index = i
            print(f"Quality dropped below {THRESHOLD} at position {i}. Cutting here.")
            break # Stop searching once we find the first bad base

Quality dropped below 20 at position 46. Cutting here.


In [12]:
# 4. Trim the record using standard Python slicing
trimmed_record = record[:cut_index]

print("\n--- RESULTS ---")
print(f"Original Length: {len(record)} bp")
print(f"Trimmed Length:  {len(trimmed_record)} bp")
print(f"Trimmed Sequence: {trimmed_record.seq}")
print(f"Trimmed Quals:    {trimmed_record.letter_annotations['phred_quality']}")


--- RESULTS ---
Original Length: 51 bp
Trimmed Length:  46 bp
Trimmed Sequence: ATGCGTACGTTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGC
Trimmed Quals:    [40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 20, 20, 20, 20, 20, 20]


In [13]:
# 5. Save the cleaned record to a new file
# (In a real script, you'd collect these in a list or use a generator)
with open("cleaned_reads.fastq", "w") as out_handle:
    SeqIO.write(trimmed_record, out_handle, "fastq")

print("\nCleaned FASTQ file saved successfully!")


Cleaned FASTQ file saved successfully!
